# Peacocks sweep: λ × motion, with evolution videos

Runs *Garden with Peacocks* (10.6 MP) through the constant-motion
annealer for every (λ, motion) combination below — **12,000 sweeps
each** — and saves a 30-second evolution video plus the full-resolution
final PNG per config, downloading each as it completes so nothing is
lost if the runtime disconnects.

**Runtime → Change runtime type → T4 GPU**, then Run all.

On GPU occupancy, honestly: at 10.6 MP a *single* run already keeps a
T4 well fed (the convolutions and 3M-proposal sweeps are large), so the
sweep runs configs sequentially — batching replicas mainly pays off for
small images. Two things here do squeeze out the remaining idle: the
code pulls the latest optimizations (stale-potential reuse, no per-sweep
syncs, on-device frame downscale), and a self-test cell below tries
`torch.compile` CUDA-graph capture and keeps it only if it works.

In [ ]:
!nvidia-smi -L || echo "NO GPU - Runtime > Change runtime type > T4 GPU"
import torch; print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# setup
import os
if not os.path.isdir('repo'):
    !git clone -q --branch claude/brave-newton-xsqgpe https://github.com/ardila/paintingReorganize.git repo
%cd repo
!git pull -q
!pip -q install scipy pillow imageio imageio-ffmpeg
import importlib, numpy as np, time
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
import smooth_palette_gpu as G
importlib.reload(G)
import matplotlib.pyplot as plt
print("ready")

In [ ]:
# download Peacocks robustly (Wikimedia rejects generic user-agents)
import os, requests
if not os.path.exists('peacocks.jpg') or open('peacocks.jpg','rb').read(3) != b'\xff\xd8\xff':
    url = ("https://commons.wikimedia.org/wiki/Special:FilePath/"
           "Franti%C5%A1ek_Kupka_%E2%80%93_Garden_with_Peacocks.jpg?width=3609")
    r = requests.get(url, headers={'User-Agent':
         'paintingReorganize-colab/1.0 (art project)'}, timeout=120)
    r.raise_for_status()
    assert r.content[:3] == b'\xff\xd8\xff', f"not a JPEG: {r.content[:80]!r}"
    open('peacocks.jpg','wb').write(r.content)
rgb = np.asarray(Image.open('peacocks.jpg').convert('RGB'))
print(f"{rgb.shape[1]}x{rgb.shape[0]} = {rgb.shape[0]*rgb.shape[1]/1e6:.1f} MP")

## Optional: `torch.compile` self-test

Static shapes make the sweep CUDA-graph capturable. This cell tries it
on a small crop and keeps it only if it runs; on any error it falls back
to eager. Skippable.

In [ ]:
USE_COMPILE = 'eager'
small = rgb[:512, :512].copy()
orig_sweep = G.sweep
import traceback
for mode in ('reduce-overhead', 'default'):
    try:
        G.sweep = torch.compile(orig_sweep, mode=mode)
        _ = G.run_constant_motion(small, sweeps=30, motion=0.005, verbose=False)
        USE_COMPILE = mode
        print(f"torch.compile mode='{mode}' works - keeping it")
        break
    except Exception:
        print(f"--- torch.compile mode='{mode}' failed:")
        traceback.print_exc(limit=3)
        G.sweep = orig_sweep
        torch._dynamo.reset()
if USE_COMPILE == 'eager':
    print("falling back to eager - correct, just without kernel fusion")


## Calibration: know the cost before you spend it

Measures ms/sweep at full resolution and prints the projected time for
the whole grid. **Trim `LAMBDAS`/`MOTIONS` in the next cell if the total
is more than your session can hold** (free Colab sessions die after a
few hours).

In [ ]:
LAMBDAS = (6.0, 12.0, 25.0)
MOTIONS = (0.002, 0.005, 0.015)
SWEEPS  = 12000

t = time.time()
_ = G.run_constant_motion(rgb, sweeps=40, motion=0.005, verbose=False)
per = (time.time() - t) / (40 + 300)     # includes the settle tail
n_cfg = len(LAMBDAS) * len(MOTIONS)
per_run_min = per * (SWEEPS + 300) / 60
print(f"~{per*1000:.0f} ms/sweep at full resolution")
print(f"one config  : ~{per_run_min:.0f} min")
print(f"grid of {n_cfg}: ~{per_run_min*n_cfg/60:.1f} hours")

## The sweep

Each config: 12,000 sweeps, a 30 s / 900-frame video (frames downscaled
4x on-device before transfer), the full-res final PNG, both downloaded
on completion. Files are named `peacocks_lam{λ}_m{motion}`.

In [ ]:
import imageio.v2 as imageio
from google.colab import files

FPS, SECONDS = 30, 30
N_FRAMES = FPS * SECONDS
FRAME_EVERY = max(1, (SWEEPS + 300) // N_FRAMES)

results = []
for lam in LAMBDAS:
    for motion in MOTIONS:
        tag = f"lam{lam:g}_m{motion:g}"
        print(f"=== {tag}", flush=True)
        writer = imageio.get_writer(f'peacocks_{tag}.mp4', fps=FPS,
                                    quality=8, macro_block_size=1)
        state = {'n': 0}
        def grab(i, frame, w=writer, st=state):
            if st['n'] < N_FRAMES:
                w.append_data(frame); st['n'] += 1
        t = time.time()
        out = G.run_constant_motion(rgb, sweeps=SWEEPS, lam=lam,
                                    motion=motion, frame_every=FRAME_EVERY,
                                    frame_scale=4, on_frame=grab,
                                    log_every=2000)
        writer.close()
        Image.fromarray(out).save(f'peacocks_{tag}.png')
        el = (time.time()-t)/60
        print(f"  {tag}: {el:.1f} min, {state['n']} frames", flush=True)
        results.append((lam, motion, out))
        files.download(f'peacocks_{tag}.mp4')
        files.download(f'peacocks_{tag}.png')

## Contact sheet of all finals

In [ ]:
rows, cols = len(LAMBDAS), len(MOTIONS)
fig, ax = plt.subplots(rows, cols, figsize=(6*cols, 5*rows))
for i, (lam, motion, out) in enumerate(results):
    a = ax.flat[i]
    a.imshow(out[::4, ::4])
    a.set_title(f'lam={lam:g}  motion={motion:g}')
    a.axis('off')
plt.tight_layout(); plt.savefig('peacocks_contact_sheet.png', dpi=110)
plt.show()
files.download('peacocks_contact_sheet.png')

## What to look for

- **motion** (row-to-row): 0.002 is the gentlest sculpt — the seed
  survives almost intact; 0.015 reorganises much more. Watch the videos:
  the T column in the logs shows the emergent temperature each target
  demands.
- **λ** (column-to-column): 6 lets the pairwise physics dominate
  (organic, blobbier); 25 disciplines the layout toward the PC1/PC2
  sweep (more gradient-like, risks pinching colour regions apart —
  it did on Demoiselles).
- Check the videos for the *frontier* behaviour where the palette has
  gaps: clean advancing boundaries good, dithered mush bad.